# Capstone — mirrors your deployed research paper

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ruzaki11/Flyrank-ml-intern-tasks/blob/main/work/notebooks/capstone.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Question

*The research question and the decision it supports.*

**Research question**

Can observable content and search-performance signals be used to identify and rank pages that are candidates for content refresh, and can a machine-learning model improve upon a simple rule-based baseline?

**Decision it supports**

The analysis supports the decision of which content pages should be prioritized for refresh review. The resulting ranking can help a content team decide which pages to investigate first, rather than treating every page as equally important.

## 2. Data

*Which release, which tables, date windows, what you excluded and why. Public-safe.*

The analysis uses the provided anonymized content-level starter table. No external datasets or private client-level sources were joined to the analysis.

In [2]:
# these are the data windows
'''
90-day:
impressions_90d
clicks_90d
pageviews_90d
sessions_90d
users_90d
engaged_sessions_90d
ai_sessions_90d
scroll_events_90d

30-day:
impressions_last_30d
clicks_last_30d
sessions_last_30d

previous 30-day:
impressions_prev_30d
clicks_prev_30d
sessions_prev_30d
'''

'\n90-day:\nimpressions_90d\nclicks_90d\npageviews_90d\nsessions_90d\nusers_90d\nengaged_sessions_90d\nai_sessions_90d\nscroll_events_90d\n\n30-day:\nimpressions_last_30d\nclicks_last_30d\nsessions_last_30d\n\nprevious 30-day:\nimpressions_prev_30d\nclicks_prev_30d\nsessions_prev_30d\n'

Features describe historical search and engagement behavior over 90-day and 30-day windows, including current and previous 30-day periods. Content freshness is represented using fields such as content age and days since the last update. The dataset does not provide a single explicit prediction timestamp for each row, so the analysis treats the supplied historical windows as the available observation period rather than claiming a specific calendar date range.

**The excluded fearures:**

1-client_id

2-content_id

Identifiers were excluded because they identify records or clients rather than representing generalizable predictive signals. client_id is retained only for grouped validation.

3- is_initial_refresh_candidate

This is the prediction target and therefore cannot be included as an input feature

4- ( needs_indexing

is_quick_win

needs_ctr_fix

needs_engagement_fix

is_underperformer

is_declining

health_score

ai_opportunity)

These fields are derived flags or scores related to content recommendations and refresh decisions. Including them could give the model information derived from the target or from existing product rules rather than requiring it to learn from the underlying signals.

5-(future_clicks

future_ctr

next_30d_sessions)

These fields describe outcomes after the prediction point and therefore would not be available when making a real refresh-prioritization decision.



## 3. Methodology

*Assumptions, features, label definition, baseline, validation design, leakage checks.*

**Assumptions**

The analysis treats is_initial_refresh_candidate as the operational definition of refresh candidacy provided in the FlyRank dataset. The objective is predictive prioritization rather than causal inference: a high predicted probability indicates that a page resembles pages labeled as refresh candidates, not that refreshing the page will necessarily improve its future performance.

Features are assumed to represent information available at or before the prediction point. Historical 30-day and 90-day performance windows are therefore treated as observational inputs, while explicitly identified future-window fields are excluded.
****

**Features**

The model uses observable content, search-performance, engagement, freshness, and trend signals. Numeric features include measures such as search volume, CPC, content age, historical impressions, clicks, sessions, CTR, average position, engagement rate, scroll rate, AI-traffic percentage, and trend percentage. Categorical features include content type, search intent, age/freshness tiers, position and impression tiers, and trend direction.

Missing numeric values are handled using median imputation, while missing categorical values are handled using the most-frequent category. Categorical variables are one-hot encoded, and numeric variables are standardized for models that require scaling. Preprocessing is fitted on the training data through a scikit-learn pipeline to avoid using test-set information during preprocessing.

****

**Label definition**

The prediction target is (is_initial_refresh_candidate), a binary field indicating whether the content record is identified as an initial refresh candidate in the supplied dataset. The target is used only as the outcome variable and is excluded from the feature matrix.
****

**Baseline**

The baseline combines (days_since_last_update) with (trend_direction) to assign a baseline score, reason code, and recommended action. The purpose of the baseline is to provide a transparent and interpretable benchmark that the machine-learning models must improve upon rather than evaluating model performance in isolation.
****

**Validation design**

Because multiple content records can belong to the same client, the model evaluation uses client-grouped validation. client_id is used as the grouping variable so that records from the same client are not intentionally distributed across the training and test sets. Stratified grouped splitting is used to preserve the binary label distribution as far as possible while maintaining client-level separation.

The model and baseline are evaluated on the same held-out records and using the same F1 metric. This makes the comparison between the rule-based baseline and the machine-learning model directly comparable.
****

**Leakage checks**

A dedicated leakage audit was performed before modeling. Fields derived from the target, existing recommendation/product flags, and future outcomes were identified and excluded from the predictive feature set. In particular, is_initial_refresh_candidate itself was excluded, along with label-derived fields such as needs_indexing, is_quick_win, needs_ctr_fix, needs_engagement_fix, is_underperformer, is_declining, health_score, and ai_opportunity. Future-window fields such as future_clicks, future_ctr, and next_30d_sessions were also excluded because they represent information that would occur after the prediction point.

Identifiers such as content_id and client_id were excluded from the model features. client_id was retained only as the grouping variable for validation.

Preprocessing was performed within the training pipeline so that imputers, encoders, and scalers were fitted using training data rather than the complete datase

****

## 4. Results (vs baseline)

*Model vs baseline on the same split. The honest table.*

The machine-learning models substantially outperformed the Week-4 rule-based baseline on the same client-grouped held-out test set. The baseline achieved an F1 score of 0.3579, while Logistic Regression and Random Forest achieved 0.7193 and 0.7192, respectively. Logistic Regression produced the highest F1 score, although the difference from Random Forest was negligible. Given the comparable predictive performance and lower model complexity, Logistic Regression was selected as the final model. The initially observed near-perfect Random Forest result was investigated and corrected before the final comparison, preventing an artificially inflated result from being used.

## 5. Limitations

*What this work cannot claim.*

****
**Observational data**

The analysis is based on historical observational data. The relationships identified by the model should therefore be interpreted as associations rather than causal effects.
****

**No guarantee of refresh impact**

A page receiving a high refresh score means that its observed characteristics resemble those associated with refresh candidacy. It does not mean that refreshing the page will necessarily increase impressions, clicks, rankings, or engagement.
****

**Historical-window limitations**

The available features summarize activity over predefined historical windows, including 30-day and 90-day periods. These aggregated windows may hide short-term changes and cannot fully describe what happened between observation periods.
****

**The model identifies pages worth investigating first; it does not determine which pages must be refreshed or prove that a refresh will improve search performance.**
****

## 6. Ranked recommendations

*The action playbook output — the paper's recommendations section.*

In [3]:
import pandas as pd

df = pd.read_csv("hf://datasets/FlyRank/internship-starter/content_refresh_anonymized.csv")

In [4]:
label = "is_initial_refresh_candidate"

y = df[label].astype(bool).astype(int)

In [5]:
from sklearn.model_selection import StratifiedGroupKFold

groups = df["client_id"]

cv = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

train_idx, test_idx = next(
    cv.split(df, y, groups=groups)
)

train_df = df.iloc[train_idx].copy()
test_df = df.iloc[test_idx].copy()

y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]

In [6]:
excluded = [
    "content_id",
    "client_id",
    "is_initial_refresh_candidate",

    # W03 leakage / label-derived fields
    "needs_indexing",
    "is_quick_win",
    "needs_ctr_fix",
    "needs_engagement_fix",
    "is_underperformer",
    "is_declining",
    "health_score",
    "ai_opportunity",

    # future-window fields, if present
    "future_clicks",
    "future_ctr",
    "next_30d_sessions",
]

feature_columns = [
    col for col in df.columns
    if col not in excluded
]

In [7]:
from numpy import select
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression


numeric_features = df[feature_columns].select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

categorical_features = df[feature_columns].select_dtypes(
    include=["object", "bool"]
).columns.tolist()



numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features)
])

L_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LogisticRegression(max_iter=1000))
])

In [8]:
X_train = train_df[numeric_features + categorical_features]
X_test = test_df[numeric_features + categorical_features]

L_model.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('numeric',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['search_volume',
                                                   'competition', 'cpc',
                                                   'word_count', 'char_count',
                                                   'impressions_90d',
                                                   'clicks_90d',
                                                   'pageviews_90d',
                                                   'sessions_90d', 'users_90d',
                                                   'engaged_sessions_90d',
                                                   'ai_sessions_90d',
                                                   'scrol...
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('encoder',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['competition_level',
                                                   'content_type',
                                                   'main_intent',
                                                   'provider_used',
                                                   'model_used', 'age_tier',
                                                   'freshness_tier',
                                                   'word_count_tier',
                                                   'char_count_tier',
                                                   'impression_tier',
                                                   'position_tier',
                                                   'trend_direction'])])),
                ('model', LogisticRegression(max_iter=1000))])

In [9]:
from sklearn.metrics import f1_score

y_pred = L_model.predict(X_test)

f1 = f1_score(y_test, y_pred)

print("Logistic Regression F1:", f1)

Logistic Regression F1: 0.7192771084337349


In [10]:
test_scores = L_model.predict_proba(X_test)[:, 1]
recommendations = test_df[["content_id"]].copy()
recommendations["score"] = test_scores

In [11]:
recommendations = (
    recommendations
    .sort_values("score", ascending=False)
    .reset_index(drop=True)
)

recommendations["rank"] = recommendations.index + 1

In [12]:
recommendations = recommendations.merge(
    test_df[
        [
            "content_id",
            "days_since_last_update",
            "trend_direction",
            "avg_position",
            "ctr",
            "impressions_90d"
        ]
    ],
    on="content_id",
    how="left"
)

In [13]:
def get_reason(row):

    if (
        row["days_since_last_update"] > 180
        and row["trend_direction"] == "down"
    ):
        return "STALE_DECLINING"

    elif row["days_since_last_update"] > 180:
        return "STALE"

    elif row["trend_direction"] == "down":
        return "DECLINING"

    else:
        return "MODEL_PRIORITY"

In [14]:
recommendations["reason_code"] = recommendations.apply(
    get_reason,
    axis=1
)

In [15]:
def get_action(reason):

    if reason == "STALE_DECLINING":
        return "Refresh Now"

    elif reason == "STALE":
        return "Review for Refresh"

    elif reason == "DECLINING":
        return "Investigate Decline"

    else:
        return "Review"

In [16]:
recommendations["action"] = (
    recommendations["reason_code"].map(get_action)
)

In [17]:
final_recommendations = recommendations[
    [
        "rank",
        "content_id",
        "score",
        "reason_code",
        "action"
    ]
]

In [18]:
final_recommendations.head(10)

,rank,content_id,score,reason_code,action
0,1,content_f7c1084c2dee,0.978559,DECLINING,Investigate Decline
1,2,content_9b5722c5285b,0.978547,DECLINING,Investigate Decline
2,3,content_eb5a338e4bb6,0.971922,DECLINING,Investigate Decline
3,4,content_e3ce13eccdf8,0.970886,DECLINING,Investigate Decline
4,5,content_a3896a2314cf,0.970546,DECLINING,Investigate Decline
5,6,content_c591436529aa,0.965962,DECLINING,Investigate Decline
6,7,content_c66416641e0c,0.964851,DECLINING,Investigate Decline
7,8,content_c89e3b5466ba,0.963698,DECLINING,Investigate Decline
8,9,content_c002cda60c0f,0.963387,DECLINING,Investigate Decline
9,10,content_144893be8d48,0.961818,DECLINING,Investigate Decline


In [19]:
top10_review = (
    final_recommendations.head(10)
    .merge(
        test_df[
            [
                "content_id",
                "days_since_last_update",
                "trend_direction",
                "trend_pct",
                "avg_position",
                "ctr",
                "impressions_90d"
            ]
        ],
        on="content_id",
        how="left"
    )
)

top10_review

,rank,content_id,score,reason_code,action,days_since_last_update,trend_direction,trend_pct,avg_position,ctr,impressions_90d
0,1,content_f7c1084c2dee,0.978559,DECLINING,Investigate Decline,20,down,-63.4,10.2,0.14,2084
1,2,content_9b5722c5285b,0.978547,DECLINING,Investigate Decline,20,down,-59.8,10.5,0.10,2979
2,3,content_eb5a338e4bb6,0.971922,DECLINING,Investigate Decline,20,down,-65.8,11.4,0.29,1043
3,4,content_e3ce13eccdf8,0.970886,DECLINING,Investigate Decline,20,down,-61.8,10.6,0.50,2779
4,5,content_a3896a2314cf,0.970546,DECLINING,Investigate Decline,20,down,-63.5,12.1,0.14,2195
5,6,content_c591436529aa,0.965962,DECLINING,Investigate Decline,20,down,-55.2,10.2,0.00,537
6,7,content_c66416641e0c,0.964851,DECLINING,Investigate Decline,20,down,-69.7,13.4,0.05,2036
7,8,content_c89e3b5466ba,0.963698,DECLINING,Investigate Decline,20,down,-47.5,11.8,0.00,2321
8,9,content_c002cda60c0f,0.963387,DECLINING,Investigate Decline,20,down,-38.2,11.6,0.00,727
9,10,content_144893be8d48,0.961818,DECLINING,Investigate Decline,20,down,-64.1,11.1,0.23,2201


In [20]:
import os

os.makedirs("work/outputs", exist_ok=True)

output_path = "work/outputs/capstone_ranked_recommendations.csv"

final_recommendations.to_csv(
    output_path,
    index=False
)

print(f"Saved to: {output_path}")

Saved to: work/outputs/capstone_ranked_recommendations.csv


## 7. Artifacts the paper embeds

*Generate/collect the charts and tables your deployed page will show.*

**Reproducibility**

 The analysis is implemented in the project repository using executable Jupyter notebooks. The workflow includes the Week-3 leakage audit, Week-4 rule-based baseline, Week-5 model development, and the capstone recommendation pipeline. The final workflow recreates the documented feature preparation, client-grouped validation design, Logistic Regression model, and ranked recommendation process. Generated outputs are stored in the repository's work/outputs/ directory. The repository and notebooks provide the implementation needed to inspect the analysis and reproduce the reported results using the available anonymized dataset.
 ****

**Output**

work/outputs/capstone_ranked_recommendations.csv

Built on the FlyRank ML Internship dataset

"hf://datasets/FlyRank/internship-starter/content_refresh_anonymized.csv"

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.